# Section 1. Install deps


In [3]:
!pip install pandas numpy gdown stanza jsonschema

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 23.7 MB/s eta 0:00:00


# Section 2. Data access


In [2]:
import pandas as pd
import gdown
import ast

file_id = "1Nv079uUCOAsO-cVQQ_UflV5xy9nkgZ6O"

url = f"https://drive.google.com/uc?id={file_id}"

output = "processed_v2.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df["text"] = df["text"].apply(lambda x: ast.literal_eval(x)["clean"])

df.head(5)

Downloading...
From: https://drive.google.com/uc?id=1Nv079uUCOAsO-cVQQ_UflV5xy9nkgZ6O
To: /content/processed_v2.csv
100%|██████████| 4.21M/4.21M [00:00<00:00, 101MB/s]


,title,text,category_id
0,Івано-Франківський драмтеатр знайшов архівні д...,на пресконференції гендиректор-художній керівн...,0
1,Премія Олеся Гончара оголосила номінантів,"Держмистецтв. Цьогоріч на здобуття премії, яку...",0
2,Біографічний фільм «Я граю Роккі» вийде у лист...,The Hollywood Reporter. Стрічка від кіностуді...,0
3,Netflix підписав угоду на трансляцію фільмів S...,"Deadline. Зазначається, що найважливішим момен...",0
4,На Венеційській бієнале Україну представить ск...,розповіли учасники Венеційської бієнале від Ук...,0


# Section 3. Extraction task definition


**Кейс:** витяг інформації про економічні/бізнес-події з новин

Потрібно отримати:
* `company` (організація/компанія)
* `event_type` (тип події)
* `amount` (сума угоди/інвестиції)
* `currency` (валюта)
* `date` (дата події)
* `location` (локація)

# Section 4. JSON schema design


**Формальна JSON schema:**

In [1]:
extraction_schema = {
  "type": "object",
  "properties": {
    "company": {
      "type": "string",
      "description": "Назва компанії або організації"
    },
    "event_type": {
      "type": "string",
      "enum": ["acquisition", "investment", "partnership", "launch", "sanctions", "other"],
      "description": "Тип події"
    },
    "amount": {
      "type": ["number", "null"],
      "description": "Сума (нормалізоване число без тексту)"
    },
    "currency": {
      "type": ["string", "null"],
      "enum": ["USD", "EUR", "UAH", None],
      "description": "Валюта"
    },
    "date": {
      "type": ["string", "null"],
      "format": "date",
      "description": "Дата у форматі YYYY-MM-DD"
    },
    "location": {
      "type": ["string", "null"],
      "description": "Локація події"
    }
  },
  "required": ["company", "event_type"],
  "additionalProperties": False
}

**Приклад:**
```
{
  "company": "Netflix",
  "event_type": "acquisition",
  "amount": 82700000000,
  "currency": "USD",
  "date": "2025-12-05",
  "location": null
}
```

# Section 5. Evaluation set


In [2]:
evaluation_set =  [
  {
      "text": "Netflix підписав угоду на трансляцію фільмів Sony Pictures після їхнього виходу у прокат. Зазначається, що найважливішим моментом в угоді на суму понад $7 мільярдів є закордонна частина. Фільми Sony більше не розподілятимуться між кількома платформами в межах закордонного вікна Pay-1, а спочатку будуть ексклюзивно виходити саме на Netflix після їхнього прокату.",
      "expected": {
         "company": "Sony Pictures",
         "event_type": "partnership",
         "amount": 7000000000,
         "currency": "USD"
      }
  },
  {
      "text": "Netflix домовилася про придбання студій та стрімінгового бізнесу Warner Bros. Discovery за $72 млрд – одного із найцінніших активів Голлівуду. Про це оголошено у пятницу, 5 грудня 2025 року. Угода може суттєво переформатувати глобальний медіаринок та викликати жорстку антимонопольну реакцію.",
      "expected": {
         "company": "Netflix",
         "event_type": "acquisition",
         "amount": 72000000000,
         "currency": "USD",
         "date": "2025-12-05"
      }
  },
  {
      "text": "Американська корпорація Google інвестує 5,5 мільярда євро у Німеччину протягом наступних чотирьох років. Це найбільша інвестиція компанії у ФРН на сьогодні. Інвестиція дозволить забезпечити у ФРН близько дев'яти тисяч робочих місць щорічно. Крім того, компанія Google офіційно оголосила про мету мінімізувати вплив своїх хмарних сервісів на клімат і довкілля в довгостроковій перспективі.",
      "expected": {
         "company": "Google",
         "event_type": "investment",
         "amount": 5500000000,
         "currency": "EUR",
         "location": "Німеччина"
      }
  },
  {
      "text": "17 березня 2026 року Apple представила iPhone 17e. Він замінює 16e і тепер пропонує вдвічі більше пам’яті, швидшу роботу і підтримку MagSafe. Крім чорного та білого, з’явився новий ніжно-рожевий колір.",
      "expected": {
         "company": "Apple",
         "event_type": "launch",
         "date": "2026-03-17"
      }
  },
  {
      "text": "В Ужгороді відкрили ювілейну виставку до 95-річчя Володимира Микити. Як розповів на відкритті виставки голова Закарпатської організації Національної спілки художників України, народний художник України Борис Кузьма - Володимир Микита був містком між основоположниками Закарпатської художньої школи та спадкоємцями.",
      "expected": {
         "company": "Закарпатська організація Національної спілки художників України",
         "event_type": "other",
         "location": "Ужгород"
      }
  },
  {
      "text": "Microsoft та OpenAI оголосили про розширення партнерства з багаторічними багатомільярдними інвестиціями, щоб швидше реалізовувати прориви в галузі штучного інтелекту (ІІ), йдеться у заяві OpenAI. Партнери повідомили, що хочуть поділитися перевагами з усім світом: розробники та організації матимуть доступ до найкращих інструментів через Azure, щоб допомогти створювати та запускати свої програми.",
      "expected": {
         "company": "Microsoft",
         "event_type": "partnership"
      }
  },
  {
      "text": "Україна запровадила санкції проти 91 морського судна російського тіньового флоту. Зазначається, що ці судна використовувалися для транспортування нафти й нафтопродуктів із російських портів до третіх країн.",
      "expected": {
         "company": "Україна",
         "event_type": "sanctions"
      }
  },
  {
      "text": "Amazon інвестує додаткові $5 млрд в Anthropic, що зміцнює партнерство в умовах посилення конкуренції у сфері штучного інтелекту. Угода також підкреслює зростаючу потребу Anthropic у обчислювальних потужностях для розвитку нових версій Claude. Як і OpenAI, компанія укладає угоди для доступу до чипів і хмарних ресурсів. Минулого тижня Anthropic оголосила про співпрацю з Broadcom для постачання чипів на базі тензорних процесорів Google — конкурента Amazon Trainium.",
      "expected": {
         "company": "Amazon",
         "event_type": "investment",
         "amount": 5000000000,
         "currency": "USD"
      }
  },
  {
      "text": "Meta Platforms представила у США оновленого помічника Meta AI, створеного на базі її великої мовної моделі Llama 3, йдеться у повідомленні Meta. У такий спосіб компанія Марка Цукерберга намагається наздогнати лідера ринку – OpenAI. Наразі будуть доступні дві версії – Llama 3 8B та Llama 3 70B, які містять 8 млрд і 70 млрд параметрів відповідно. Моделі будуть інтегровані у віртуальний помічник Meta AI, який було презентовано у вересні минулого року.",
      "expected": {
         "company": "Meta",
         "event_type": "launch",
         "location": "США"
      }
  },
  {
      "text": "Європарламент схвалив резолюцію про стратегічну оборону ЄС та партнерства у сфері безпеки, у якій Україну назвали пріоритетним стратегічним партнером і запропонували формалізувати співпрацю з нею у сфері безпеки та оборони.",
      "expected": {
         "company": "Європарламент",
         "event_type": "other",
         "location": "Україна"
      }
  },
  {
      "text": "Із початку повномасштабної війни МВФ надав Україні понад $12,4 мільярда - Шмигаль. Прем’єр-міністр України Денис Шмигаль обговорив із головою Європейського департаменту МВФ Альфредом Каммером деталі подальшої співпраці.",
      "expected": {
         "company": "МВФ",
         "event_type": "investment",
         "amount": 12400000000,
         "currency": "USD"
      }
  },
  {
      "text": "ООН разом із партнерами оголосила масштабний План гуманітарних потреб та реагування для України на 2026 рік, спрямований на допомогу найбільш уразливим верствам населення, які постраждали від повномасштабної війни.",
      "expected": {
         "company": "ООН",
         "event_type": "other",
         "location": "Україна"
      }
  },
  {
      "text": "Студія Warner Bros інвестує 1,25 млрд доларів у будівництво повністю функціонального «міні-міста», яке стане декораціями до нового серіалу про Гаррі Поттера. Творці серіалу планують побудувати повноцінне містечко Літтл-Уінгінг, з усім необхідним для організації виробництва - новими дорогами, багатоповерховими парковками, павільйонами для зйомок та безліччю допоміжних об'єктів.",
      "expected": {
         "company": "Warner Bros",
         "event_type": "investment",
         "amount": 1250000000,
         "currency": "USD"
      }
  },
  {
      "text": "Як повідомив CNN, фільм не мав успіху в жодному американському місті в день прем\'єри. Деякі кінотеатри були майже порожні..",
      "expected": {
         "company": "CNN",
         "event_type": "other"
      }
  },
  {
      "text": "Компанія SpaceX запустила 24 супутники Starlink з космічної бази Ванденберг у Каліфорнії. Їх на орбіту доставила ракета-носій Falcon 9, це був її 22-й політ. Після розділення один зі ступенів ракети приземлився на безпілотник Of Course I Still Love You у Тихому океані. Супутники Starlink запускають на орбіту у межах створення глобальної космічної мережі для роботи високошвидкісного супутникового інтернету у будь-якій точці планети.",
      "expected": {
         "company": "SpaceX",
         "event_type": "launch",
         "location": "Каліфорнія"
      }
  },
  {
      "text": "Європарламент схвалив надання Україні кредиту на €90 мільярдів. Відповідне рішення підтримали 458 депутатів, 140 були ""проти"" (44 - утрималися). З цієї суми 30 мільярдів євро буде виділено на макрофінансову допомогу або бюджетну підтримку, що надаватиметься через Механізм ЄС для України. 60 млрд євро буде виділено на зміцнення оборонного потенціалу України та підтримку закупівель військового обладнання. Якщо певні оборонні матеріали не будуть негайно доступні з європейських країн, то для їх постачання з ""третіх"" держав буде застосовано низку цільових відступів.",
      "expected": {
         "company": "Європарламент",
         "event_type": "investment",
         "amount": 90000000000,
         "currency": "EUR",
         "location": "Україна"
      }
  }
]

# Section 6. Baseline prompt


In [2]:
text_to_process = "Американська корпорація Google інвестує 5,5 мільярда євро у Німеччину протягом наступних чотирьох років."

prompt = [
      {"role": "system", "content": "You are a precise information extraction system. Respond ONLY with valid JSON."},
      {"role": "user", "content": f"""Extract data from the text below.

  Rules:
  - If missing → null
  - Use YYYY-MM-DD for dates if possible
  - Amount must be a number
  - Extract only the core brand name
  - event_type must be one of ["acquisition", "investment", "partnership", "launch", "sanctions", "other"]
  - Normalize currency (e.g., "доларів" → USD, "€" -> EUR, "$" -> USD)
  - Choose the main company which mentioned first if multiple are mentioned


  Output schema:
  {{
    "company": string,
    "event_type": lowercase string,
    "amount": number or null,
    "currency": string or null,
    "date": string or null,
    "location": string or null
  }}

  Text: {text_to_process}"""}
  ]

# Section 7. Raw extraction


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cpu"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [9]:
results = []

for item in evaluation_set[:10]:
  messages = [
      {"role": "system", "content": "You are a precise information extraction system. Respond ONLY with valid JSON."},
      {"role": "user", "content": f"""Extract data from the text below.

  Rules:
  - If missing → null
  - Use YYYY-MM-DD for dates if possible
  - Amount must be a number
  - Extract only the core brand name
  - event_type must be one of ["acquisition", "investment", "partnership", "launch", "sanctions", "other"]
  - Normalize currency (e.g., "доларів" → USD, "€" -> EUR, "$" -> USD)
  - Choose the main company which mentioned first if multiple are mentioned


  Output schema:
  {{
    "company": string,
    "event_type": lowercase string,
    "amount": number or null,
    "currency": string or null,
    "date": string or null,
    "location": string or null
  }}

  Text: {item['text']}"""}
  ]


  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  outputs = pipe(prompt, max_new_tokens=256, do_sample=False, return_full_text=False)
  generated_text = outputs[0]["generated_text"]
  results.append(generated_text)

  print(generated_text)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "company": "Sony Pictures",
  "event_type": "acquisition",
  "amount": 7000000000,
  "currency": "USD",
  "date": "null",
  "location": "null"
}


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "company": "Warner Bros. Discovery",
  "event_type": "acquisition",
  "amount": 72000000000,
  "currency": "USD",
  "date": "2025-12-05",
  "location": null
}


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

{
  "company": "Російський тіньовый флот",
  "event_type": "sanctions",
  "amount": null,
  "currency": null,
  "date": "2023-04-05",
  "location": "морські судна"
}


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "company": "Amazon",
  "event_type": "investment",
  "amount": 5000000000,
  "currency": "USD",
  "date": "2023-01-04",
  "location": null
}


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "company": "Meta",
  "event_type": "launch",
  "amount": null,
  "currency": null,
  "date": "2021-11-04",
  "location": "США"
}
{
  "company": "Європарламент",
  "event_type": "partnership",
  "amount": null,
  "currency": null,
  "date": "2023-10-05",
  "location": null
}


# Section 8. JSON validator


In [11]:
import json
from jsonschema import validate, ValidationError

def validate_extraction_output(raw_output):
    print(f"--- Аналіз виходу моделі ---")
    try:
        data = json.loads(raw_output)
        print("Parse Success: Рядок є коректним JSON.")
    except json.JSONDecodeError as e:
        print(f"Parse Fail: Помилка синтаксису JSON: {e}")
        return False

    try:
        validate(instance=data, schema=extraction_schema)
        print("Schema Success: Дані повністю відповідають схемі.")
        return True
    except ValidationError as e:
        print(f"Schema Fail: Невідповідність схемі!")
        print(f"{e.message}")
        return False

for item in results:
  validate_extraction_output(item)

--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коректним JSON.
Schema Success: Дані повністю відповідають схемі.
--- Аналіз виходу моделі ---
Parse Success: Рядок є коре

# Section 9. Repair loop


In [20]:
import json
from jsonschema import validate, ValidationError

MAX_REPAIR_ATTEMPTS = 2

def validate_output(text):

    try:
        data = json.loads(text.strip())
        validate(instance=data, schema=extraction_schema)
        return True, data, None
    except json.JSONDecodeError as e:
        return False, None, f"Помилка парсингу JSON: {str(e)}"
    except ValidationError as e:
        return False, None, f"Помилка валідації схеми: {e.message}"
    except Exception as e:
        return False, None, f"Непередбачена помилка: {str(e)}"


def run_extraction_with_repair(initial_messages):

    current_messages = list(initial_messages)
    attempt = 0

    while attempt <= MAX_REPAIR_ATTEMPTS:
        prompt = tokenizer.apply_chat_template(
            current_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        outputs = pipe(
            prompt,
            max_new_tokens=256,
            do_sample=False,
            return_full_text=False
        )

        generated_text = outputs[0]["generated_text"].strip()

        is_valid, data, error_msg = validate_output(generated_text)

        if is_valid:
            print(f"Успішно на спробі {attempt}")
            return {"status": "success", "data": data, "attempts": attempt}

        print(f"Спроба {attempt} провалена: {error_msg}")

        if attempt < MAX_REPAIR_ATTEMPTS:
            current_messages.append({"role": "assistant", "content": generated_text})

            repair_instruction = (
                f"Your previous result contains an error: {error_msg}. "
                f"Please fix the error and return ONLY a valid JSON object that "
                f"matches to the schema."
            )
            current_messages.append({"role": "user", "content": repair_instruction})

        attempt += 1

    return {
        "status": "failed",
        "last_output": generated_text,
        "error": error_msg,
        "attempts": MAX_REPAIR_ATTEMPTS
    }

results_repair = []

for item in evaluation_set[:10]:
    messages = [
      {"role": "system", "content": "You are a precise information extraction system. Respond ONLY with valid JSON."},
      {"role": "user", "content": f"""Extract data from the text below.

    Rules:
    - If missing → null
    - Use YYYY-MM-DD for dates if possible
    - Amount must be a number
    - Extract only the core brand name
    - event_type must be one of ["acquisition", "investment", "partnership", "launch", "sanctions", "other"]
    - Normalize currency (e.g., "доларів" → USD, "€" -> EUR, "$" -> USD)
    - Choose the main company which mentioned first if multiple are mentioned


    Output schema:
    {{
      "company": string,
      "event_type": lowercase string,
      "amount": number or null,
      "currency": string or null,
      "date": string or null,
      "location": string or null
    }}

    Text: {item['text']}"""}
    ]

    final_result = run_extraction_with_repair(messages)
    results_repair.append(final_result)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Успішно на спробі 0


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Успішно на спробі 0


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Успішно на спробі 0


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Успішно на спробі 0


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Успішно на спробі 0
Успішно на спробі 0


# Section 10. Metrics: valid JSON rate


In [26]:
import json
from jsonschema import validate, ValidationError

def calculate_repair_metrics(results_repair):
    total_examples = len(results_repair)
    if total_examples == 0:
        return "Немає даних для розрахунку"

    raw_valid_json_count = 0
    post_repair_valid_count = 0
    schema_valid_count = 0
    repair_needed_count = 0
    repair_failed_count = 0
    total_repairs_made = 0

    for res in results_repair:
        if res['status'] == 'success' and res['attempts'] == 0:
            raw_valid_json_count += 1

        if res['status'] == 'success':
            post_repair_valid_count += 1
            schema_valid_count += 1

        if res['attempts'] > 0:
            repair_needed_count += 1
            total_repairs_made += res['attempts']

        if res['status'] == 'failed':
            repair_failed_count += 1

    raw_rate = (raw_valid_json_count / total_examples) * 100
    post_repair_rate = (post_repair_valid_count / total_examples) * 100
    schema_valid_rate = (schema_valid_count / total_examples) * 100

    need_repair_rate = (repair_needed_count / total_examples) * 100
    repair_helped_rate = 0
    if repair_needed_count > 0:
        helped_count = post_repair_valid_count - raw_valid_json_count
        repair_helped_rate = (helped_count / repair_needed_count) * 100

    avg_repairs = total_repairs_made / total_examples

    print("--- ОСНОВНІ МЕТРИКИ ---")
    print(f"1. Raw valid JSON rate (до repair): {raw_rate:.2f}%")
    print(f"2. Post-repair valid JSON rate: {post_repair_rate:.2f}%")
    print(f"3. Schema-valid JSON rate: {schema_valid_rate:.2f}%")

    print("\n--- ДОДАТКОВІ МЕТРИКИ ---")
    print(f"Average repairs per example: {avg_repairs:.2f}")
    print(f"% прикладів, де repair був потрібен: {need_repair_rate:.2f}%")
    print(f"% прикладів, де repair допоміг: {repair_helped_rate:.2f}%")
    print(f"% прикладів, де repair НЕ допоміг: {(repair_failed_count/total_examples)*100:.2f}%")

calculate_repair_metrics(results_repair)

--- ОСНОВНІ МЕТРИКИ ---
1. Raw valid JSON rate (до repair): 100.00%
2. Post-repair valid JSON rate: 100.00%
3. Schema-valid JSON rate: 100.00%

--- ДОДАТКОВІ МЕТРИКИ ---
Average repairs per example: 0.00
% прикладів, де repair був потрібен: 0.00%
% прикладів, де repair допоміг: 0.00%
% прикладів, де repair НЕ допоміг: 0.00%


# Section 11. Error analysis


In [25]:
import json
from jsonschema import validate, ValidationError

def analyze_and_compare(raw_results, repair_results, schema):
    def get_detailed_status(text):
        try:
            data = json.loads(text.strip())
            try:
                validate(instance=data, schema=schema)
                return "valid"
            except ValidationError:
                return "schema_error"
        except (json.JSONDecodeError, AttributeError):
            return "parse_error"

    stats = {
        "Variant 1 (Raw)": {"valid": 0, "parse_error": 0, "schema_error": 0},
        "Variant 2 (Repair)": {"valid": 0, "parse_error": 0, "schema_error": 0}
    }

    for raw_output in raw_results:
        status = get_detailed_status(raw_output)
        stats["Variant 1 (Raw)"][status] += 1

    for res in repair_results:
        if res['status'] == 'success':
            stats["Variant 2 (Repair)"]["valid"] += 1
        else:
            status = get_detailed_status(res['last_output'])
            stats["Variant 2 (Repair)"][status] += 1

    total = len(raw_results)

    def print_metrics(label, data):
        valid_rate = (data["valid"] / total) * 100
        parse_fail_rate = (data["parse_error"] / total) * 100
        schema_fail_rate = (data["schema_error"] / total) * 100
        print(f"\n--- {label} ---")
        print(f"Valid JSON Rate: {valid_rate:.2f}%")
        print(f"Parse Failures (Syntax): {parse_fail_rate:.2f}%")
        print(f"Schema Failures (Structure): {schema_fail_rate:.2f}%")

    print_metrics("Варіант 1 — Raw Extraction", stats["Variant 1 (Raw)"])
    print_metrics("Варіант 2 — Extraction + Repair Loop", stats["Variant 2 (Repair)"])

    fixed_by_repair = stats["Variant 2 (Repair)"]["valid"] - stats["Variant 1 (Raw)"]["valid"]
    print("\n--- ЕФЕКТИВНІСТЬ REPAIR LOOP ---")
    print(f"Кількість кейсів, які врятував Repair Loop: {fixed_by_repair}")
    if fixed_by_repair > 0:
        improvement = (fixed_by_repair / total) * 100
        print(f"Абсолютне покращення точності: +{improvement:.2f}%")

    print("\n--- ВИСНОВКИ ---")
    print("1. Де Repair Loop реально допомагає:")
    print("   - Пропущені коми або зайві лапки (Syntax).")
    print("   - Неправильний тип даних (напр. рядок замість числа).")
    print("   - Відсутність обов'язкових полів (Schema).")
    print("\n2. Де Repair Loop не рятує:")
    print("   - Галюцинації: якщо моделі немає в тексті даних, вона може вигадувати JSON, який валідний, але фактично невірний.")
    print("   - Складні вкладені структури, які модель хронічно не розуміє.")
    print("   - Якщо є ліміт токенів (max_new_tokens) обриває JSON на середині.")

analyze_and_compare(results, results_repair, extraction_schema)


--- Варіант 1 — Raw Extraction ---
Valid JSON Rate: 100.00%
Parse Failures (Syntax): 0.00%
Schema Failures (Structure): 0.00%

--- Варіант 2 — Extraction + Repair Loop ---
Valid JSON Rate: 100.00%
Parse Failures (Syntax): 0.00%
Schema Failures (Structure): 0.00%

--- ЕФЕКТИВНІСТЬ REPAIR LOOP ---
Кількість кейсів, які врятував Repair Loop: 0

--- ВИСНОВКИ ---
1. Де Repair Loop реально допомагає:
   - Пропущені коми або зайві лапки (Syntax).
   - Неправильний тип даних (напр. рядок замість числа).
   - Відсутність обов'язкових полів (Schema).

2. Де Repair Loop не рятує:
   - Галюцинації: якщо моделі немає в тексті даних, вона може вигадувати JSON, який валідний, але фактично невірний.
   - Складні вкладені структури, які модель хронічно не розуміє.
   - Якщо ліміт токенів (max_new_tokens) обриває JSON на середині.


**Аналіз помилкових прикладів:**

In [ ]:
error_set = """{
   "expected": {
         "company": "Sony Pictures",
         "event_type": "partnership",
         "amount": 7000000000,
         "currency": "USD",
         "date": null,
         "location": null
      },
   "predicted": {
         "company": "Sony Pictures",
         "event_type": "acquisition",
         "amount": 7000000000,
         "currency": "USD",
         "date": "null",
         "location": "null"
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON"],
   "comment": "Неправильно визначено тип події: partnership -> acquisition"
},
{
   "expected": {
         "company": "Netflix",
         "event_type": "acquisition",
         "amount": 72000000000,
         "currency": "USD",
         "date": "2025-12-05",
         "location": null
      },
   "predicted": {
         "company": "Warner Bros. Discovery",
         "event_type": "acquisition",
         "amount": 72000000000,
         "currency": "USD",
         "date": "2025-12-05",
         "location": null
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON"],
   "comment": "Обрано неправильну компанію (Warner Bros. Discovery замість Netflix)"
},
{
   "expected": {
         "company": "Google",
         "event_type": "investment",
         "amount": 5500000000,
         "currency": "EUR",
         "date": null,
         "location": "Німеччина"
      },
   "predicted": {
         "company": "Google",
         "event_type": "investment",
         "amount": 5500000000,
         "currency": "EUR",
         "date": "2023-10-04",
         "location": "ФРН"
      },
   "valid": False,
   "error_category": ["hallucinated field/value", "normalization issue"],
   "comment": "Дата вигадана (не була в тексті); 'ФРН' замість очікуваного 'Німеччина'"
},
{
   "expected": {
         "company": "Apple",
         "event_type": "launch",
         "amount": null,
         "currency": null,
         "date": "2026-03-17",
         "location": null
      },
   "predicted": {
         "company": "Apple",
         "event_type": "launch",
         "amount": null,
         "currency": null,
         "date": "2026-03-17",
         "location": null
      },
   "valid": False,
   "error_category": ["null handling issue"],
   "comment": "У predicted значення 'null' задано як string, тоді очікується null"
},
{
   "expected": {
         "company": "Закарпатська організація Національної спілки художників України",
         "event_type": "other",
         "amount": null,
         "currency": null,
         "date": null,
         "location": "Ужгород"
      },
   "predicted": {
         "company": "Национальна сільська ради українських художників",
         "event_type": "launch",
         "amount": null,
         "currency": null,
         "date": "2018-03-06",
         "location": "Ужгород"
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON", "hallucinated field/value"],
   "comment": "Назва організації сильно спотворена; також неправильно визначено тип події та вигадано дату"
},
{
   "expected": {
         "company": "Microsoft",
         "event_type": "partnership",
         "amount": null,
         "currency": null,
         "date": null,
         "location": null
      },
   "predicted": {
         "company": "Microsoft",
         "event_type": "investment",
         "amount": null,
         "currency": null,
         "date": "2023-10-05",
         "location": null
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON", "hallucinated field/value"],
   "comment": "Тип події визначено неправильно (partnership -> investment); дата вигадана"
},
{
   "expected": {
         "company": "Україна",
         "event_type": "sanctions",
         "amount": null,
         "currency": null,
         "date": null,
         "location": null
      },
   "predicted": {
         "company": "Російський тіньовый флот",
         "event_type": "sanctions",
         "amount": null,
         "currency": null,
         "date": "2023-04-05",
         "location": "морські судна"
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON", "hallucinated field/value"],
   "comment": "Обрано неправильну сутність (не Україна); вигадано дату та некоректну локацію"
},
{
   "expected": {
         "company": "Amazon",
         "event_type": "investment",
         "amount": 5000000000,
         "currency": "USD",
         "date": null,
         "location": null
      },
   "predicted": {
         "company": "Amazon",
         "event_type": "investment",
         "amount": 5000000000,
         "currency": "USD",
         "date": "2023-01-04",
         "location": null
      },
   "valid": False,
   "error_category": ["hallucinated field/value"],
   "comment": "Дата вигадана — у тексті відсутня"
},
{
   "expected": {
         "company": "Meta",
         "event_type": "launch",
         "amount": null,
         "currency": null,
         "date": null,
         "location": "США"
      },
   "predicted": {
         "company": "Meta",
         "event_type": "launch",
         "amount": null,
         "currency": null,
         "date": "2021-11-04",
         "location": "США"
      },
   "valid": False,
   "error_category": ["hallucinated field/value"],
   "comment": "Дата вигадана — у тексті відсутня"
},
{
   "expected": {
         "company": "Європарламент",
         "event_type": "other",
         "amount": null,
         "currency": null,
         "date": null,
         "location": "Україна"
      },
   "predicted": {
         "company": "Європарламент",
         "event_type": "partnership",
         "amount": null,
         "currency": null,
         "date": "2023-10-05",
         "location": null
      },
   "valid": False,
   "error_category": ["semantic extraction error despite valid JSON", "hallucinated field/value"],
   "comment": "Неправильний тип події (other -> partnership); вигадана дата; пропущена локація"
}"""

**Аналіз помилок:**
* Extraction у загальному змістовно правильний, але має помилки
* Модель “галюцинує” у `date` та `location` - додає дату та іноді локацію, навіть якщо їх нема в тексті.
* Найчастіше ламаються поля:
1. `event_type`: модель плутає partnership, acquisition та investment
2. `company`: вибір не тієї сутності, спотворення назви
3. `location`: погана нормалізація (ФРН vs Німеччина), іноді пропуски
* Модель не пропускає поля, однак замість null іноді намагається вигадати значення
* Іноді замість null визначається стрічка "null"

**Підсумок:**
* Наймасовіші категорії помилок
1. hallucinated field/value (найчастище у `date`)
2. semantic extraction error (`event_type`, `company`)
3. normalization issues (`location`, `company`)
4. null handling issue

* Repair loop добре покриває
1. JSON формат (пропущені коми або зайві лапки, відсутність обов'язкових полів)
2. Типи (number, null)
3. Частково `currency` нормалізацію

* Проблеми, які repair loop не покиває
1. Галюцинації (якщо моделі немає в тексті даних, вона може вигадувати JSON, який валідний, але фактично невірний)
2. Semantic помилки (невірний вибір `event_type` та `company`)


# Section 12. Generate docs/audit_summary_lab11.md

In [ ]:
report = """# Audit Summary — Lab 11 ( LLM extraction як інженерія (schema-first))
## Який extraction-кейс
**Кейс:** витяг інформації про економічні/бізнес-події з новин

Потрібно отримати:
* `company` (організація/компанія)
* `event_type` (тип події)
* `amount` (сума угоди/інвестиції)
* `currency` (валюта)
* `date` (дата події)
* `location` (локація)
## Скільки прикладів у evaluation set
Evaluation set складається з 16 текстів, кожен з яких має вручну створену еталонну (gold) JSON-розмітку з полями `text` та `expected`.
## Який raw valid JSON rate
Raw valid JSON rate (до repair): 100.00%
## Який post-repair valid JSON rate
Post-repair valid JSON rate: 100.00%
## Який schema-valid JSON rate
Schema-valid JSON rate: 100.00%
## Поля, що ламаються найчастіше
1. `event_type`: модель плутає partnership, acquisition та investment
2. `company`: вибір не тієї сутності, спотворення назви
3. `location`: погана нормалізація (ФРН vs Німеччина), іноді пропуски
## Наймасовіші категорії помилок
1. hallucinated field/value (найчастище у `date`)
2. semantic extraction error (`event_type`, `company`)
3. normalization issues (`location`, `company`)
4. null handling issue
## Чи schema-first підхід спрацював добре
Schema-first підхід добре спрацював для забезпечення коректної структури та типів даних (валидний JSON майже у всіх випадках). Водночас він не вирішує змістовні помилки — модель продовжує галюцинувати та неправильно інтерпретувати події і сутності.
"""

with open("audit_summary_lab11.md", "w", encoding="utf-8") as f:
    f.write(report)